# Questions 3, 4 and 5 · SQL analysis of the original CSVs


## Definitions

- Include orders with `order_status = 'delivered'` and both delivery dates present.
- Mark an order late when its actual delivery **day** is after its estimated delivery **day**.
- A low rating is a score of 1 or 2.
- Select the latest answered review for each order. The SQL file documents the tie-breaking rules.
- Count an order once in each of its product categories. A multi-category order contributes to several category totals.
- Use item prices in BRL for question 5. Freight is kept outside this measure.

These definitions should be agreed with the team before the final analysis.

In [2]:
tables = {
    table: pd.read_csv(data_dir / filename)
    for table, filename in csv_files.items()
}
for table, key in (("orders", "order_id"), ("customers", "customer_id"),
                   ("products", "product_id"), ("category_translation", "product_category_name")):
    if tables[table][key].isna().any() or tables[table][key].duplicated().any():
        raise ValueError(f"{table}.{key} must be unique and populated.")

# Split complete SQL statements, respecting comments and quoted text.
statements = []
buffer = ""
for line in (project_dir / "queries.sql").read_text(encoding="utf-8-sig").splitlines(keepends=True):
    buffer += line
    if sqlite3.complete_statement(buffer):
        statements.append(buffer)
        buffer = ""

selected_queries = {}
for statement in statements:
    match = re.search(r"(?m)^-- QUESTION ([345])\s*$", statement)
    if match:
        selected_queries[int(match.group(1))] = statement
if set(selected_queries) != {3, 4, 5}:
    raise ValueError("Expected queries with the comments -- QUESTION 3, -- QUESTION 4 and -- QUESTION 5.")

connection = sqlite3.connect(":memory:")
try:
    for table, data in tables.items():
        data.to_sql(table, connection, index=False)
    results = {
        number: pd.read_sql_query(selected_queries[number], connection)
        for number in (3, 4, 5)
    }
finally:
    connection.close()

print(f"Orders in source: {len(tables['orders']):,}")
print(f"Eligible delivered orders: {int(results[5].iloc[0]['delivered_orders']):,}")
print(f"Reviewed eligible orders: {int(results[5].iloc[0]['reviewed_orders']):,}")

NameError: name 'csv_files' is not defined

## Question 3 · Where do delays occur?

The first table ranks states by the number of late orders. The second ranks them by their within-state delay rate and shows their order counts.

In [ ]:
states = results[3]
display(states)
display(
    states.sort_values(
        ["delayed_percentage", "delivered_orders", "customer_state"],
        ascending=[False, False, True],
    ).head(5)
)

,customer_state,delivered_orders,delayed_orders,delayed_percentage
0,SP,40494,1820,4.49
1,RJ,12350,1495,12.11
2,MG,11354,519,4.57
3,BA,3256,396,12.16
4,RS,5344,325,6.08
5,SC,3546,291,8.21
6,ES,1995,214,10.73
7,PR,4923,199,4.04
8,CE,1279,176,13.76
9,PE,1593,153,9.60


,customer_state,delivered_orders,delayed_orders,delayed_percentage
14,AL,397,85,21.41
11,MA,717,125,17.43
19,SE,335,51,15.22
16,PI,476,66,13.87
8,CE,1279,176,13.76


## Question 4 · Which categories accumulate delays or low ratings?

The tables show the leading categories by delay count and low-rating count. The accompanying percentages use eligible orders and reviewed eligible orders, respectively. Ratings are collected at order level.

In [ ]:
categories = results[4]
display(categories.head(5))
display(
    categories.sort_values(
        ["low_rating_orders", "category"], ascending=[False, True]
    ).head(5)
)

,category_portuguese,category,delivered_orders,delayed_orders,delayed_percentage,reviewed_orders,average_review_score,low_rating_orders,low_rating_percentage,delayed_low_rating_orders
0,cama_mesa_banho,bed_bath_table,9272,689,7.43,9177,4.00,1471,16.03,432
1,beleza_saude,health_beauty,8647,649,7.51,8601,4.23,985,11.45,385
2,esporte_lazer,sports_leisure,7529,495,6.57,7485,4.23,852,11.38,316
3,moveis_decoracao,furniture_decor,6307,449,7.12,6260,4.06,961,15.35,269
4,informatica_acessorios,computers_accessories,6529,417,6.39,6498,4.08,945,14.54,251


,category_portuguese,category,delivered_orders,delayed_orders,delayed_percentage,reviewed_orders,average_review_score,low_rating_orders,low_rating_percentage,delayed_low_rating_orders
0,cama_mesa_banho,bed_bath_table,9272,689,7.43,9177,4.00,1471,16.03,432
1,beleza_saude,health_beauty,8647,649,7.51,8601,4.23,985,11.45,385
3,moveis_decoracao,furniture_decor,6307,449,7.12,6260,4.06,961,15.35,269
4,informatica_acessorios,computers_accessories,6529,417,6.39,6498,4.08,945,14.54,251
2,esporte_lazer,sports_leisure,7529,495,6.57,7485,4.23,852,11.38,316


## Question 5 · Item sales associated with late, poorly rated orders

The indicator sums the prices of all items in orders that were late and received a score of 1 or 2. The percentage compares this amount with item sales across all eligible delivered orders.

In [ ]:
revenue = results[5]
display(revenue)

,delivered_orders,reviewed_orders,orders_with_item_revenue,delayed_low_rating_orders,total_item_revenue_brl,at_risk_item_revenue_brl,at_risk_revenue_percentage
0,96470,95824,96470,3983,13220248.93,619012.41,4.68


## Export the answers and complete result tables

This cell writes three CSVs and `results_questions_3_4_5.md` to `resultados/`. Include these files and the notebook in the repository to share the analysis with your teammate. Re-run the analysis after agreeing on the final cleaning and review-selection rules.

In [ ]:
from datetime import date

output_names = {
    3: "question_3_states.csv",
    4: "question_4_categories.csv",
    5: "question_5_revenue.csv",
}
for question, filename in output_names.items():
    results[question].to_csv(results_dir / filename, index=False, encoding="utf-8", float_format="%.2f")


def markdown_table(frame):
    table_lines = [
        "| " + " | ".join(frame.columns) + " |",
        "| " + " | ".join("---" for _ in frame.columns) + " |",
    ]
    for values in frame.itertuples(index=False, name=None):
        cells = ["" if pd.isna(value) else str(value) for value in values]
        table_lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(table_lines)


state_leaders = states.head(5)
state_rate_leaders = states.sort_values(
    ["delayed_percentage", "delivered_orders", "customer_state"],
    ascending=[False, False, True],
).head(5)
category_delay_leaders = categories.head(5)[
    ["category", "delivered_orders", "delayed_orders", "delayed_percentage"]
]
category_low_leaders = categories.sort_values(
    ["low_rating_orders", "category"], ascending=[False, True]
).head(5)[
    ["category", "reviewed_orders", "low_rating_orders", "low_rating_percentage", "average_review_score"]
]
metrics = revenue.iloc[0]
delivered_count = int(metrics["delivered_orders"])
reviewed_count = int(metrics["reviewed_orders"])
late_count = int(states["delayed_orders"].sum())
late_share = 100 * late_count / delivered_count
state_by_count = state_leaders.iloc[0]
state_by_rate = state_rate_leaders.iloc[0]
category_by_delay = category_delay_leaders.iloc[0]
category_by_low = category_low_leaders.iloc[0]
excluded_delivered = int(tables["orders"]["order_status"].eq("delivered").sum()) - delivered_count

report = f"""# Olist · Answers to questions 3, 4 and 5

Analysis date: {date.today().isoformat()}. Source: original CSVs from [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce), version 2.

The queries were executed in SQLite over the original CSVs. Their logic was checked against independent pandas calculations, including monetary totals checked in exact decimal arithmetic. Validation against the final MySQL database is pending.

## Scope and definitions

- The source contains {len(tables["orders"]):,} orders.
- The analysis includes {delivered_count:,} delivered orders with both delivery dates available. It excludes {excluded_delivered:,} orders marked delivered that have a missing delivery date.
- There are {late_count:,} late orders, representing {late_share:.2f}% of eligible delivered orders.
- A late order arrives on a calendar day after its estimated delivery day.
- A low rating is a score of 1 or 2.
- For orders with multiple reviews, the latest answered review is used. Ties use review creation date, review ID and score in descending order.
- {reviewed_count:,} eligible orders have a selected valid review. Missing reviews stay outside rating averages and low-rating percentages.
- Customer state comes from customers.customer_state.
- Categories use the English translation where available and the original Portuguese name otherwise. Missing categories appear as unknown.
- The category analysis counts one order per category. Multi-category orders contribute to multiple groups. The review score describes the entire order.
- Question 5 uses the sum of item prices in BRL, excluding freight.

## Question 3 · Which states have the most delayed orders?

By volume, **{state_by_count["customer_state"]}** leads with **{int(state_by_count["delayed_orders"]):,} late orders**, a delay rate of **{state_by_count["delayed_percentage"]:.2f}%** among its {int(state_by_count["delivered_orders"]):,} eligible delivered orders.

### Top five states by delay count

{markdown_table(state_leaders)}

### Top five states by delay rate

{markdown_table(state_rate_leaders)}

**{state_by_rate["customer_state"]}** has the highest delay rate, **{state_by_rate["delayed_percentage"]:.2f}%**, based on {int(state_by_rate["delayed_orders"]):,} late orders out of {int(state_by_rate["delivered_orders"]):,} eligible deliveries.

**Conclusion.** Prioritization can consider both the large number of affected orders in SP and the elevated delay rates in AL, MA and SE. Their different order volumes remain visible in the tables.

## Question 4 · Which categories accumulate delays or low ratings?

**{category_by_delay["category"]}** leads by delay count, with **{int(category_by_delay["delayed_orders"]):,} late orders** and a **{category_by_delay["delayed_percentage"]:.2f}%** delay rate.

### Top five categories by delay count

{markdown_table(category_delay_leaders)}

### Top five categories by low-rating count

{markdown_table(category_low_leaders)}

**{category_by_low["category"]}** also leads by low-rating count, with **{int(category_by_low["low_rating_orders"]):,} orders rated 1 or 2**, or **{category_by_low["low_rating_percentage"]:.2f}%** of its reviewed eligible orders.

**Conclusion.** Bed, bath and table products are a priority for investigating delivery and customer experience because they lead both counts. Health and beauty products also have a large number of delayed orders. Category-level order counts and percentages provide the context for those rankings.

## Question 5 · How much revenue is associated with late orders with low ratings?

| Measure | Value |
| --- | --- |
| Late orders rated 1 or 2 | {int(metrics["delayed_low_rating_orders"]):,} |
| Item sales in these orders | {metrics["at_risk_item_revenue_brl"]:,.2f} BRL |
| Item sales across all eligible delivered orders | {metrics["total_item_revenue_brl"]:,.2f} BRL |
| Share of eligible item sales | {metrics["at_risk_revenue_percentage"]:.2f}% |

**Conclusion.** Late orders with low ratings account for **{metrics["at_risk_item_revenue_brl"]:,.2f} BRL**, or **{metrics["at_risk_revenue_percentage"]:.2f}%** of eligible item sales. This is a descriptive indicator of sales associated with a poor delivery and review outcome. Estimating refunds, future customer spending or lost sales would require additional data and analysis.

## Reproduce and integrate

1. Keep queries.sql and consultas_csv.ipynb in the repository root, with the original CSVs in datos/.
2. Execute all cells in consultas_csv.ipynb to regenerate the three CSVs and this report in resultados/.
3. Agree the late-delivery, low-rating and review-selection rules with the teammate responsible for cleaning orders and reviews.
4. Execute queries.sql against the final MySQL tables and reconcile the results before the final presentation.

Data attribution: Olist, Brazilian E-Commerce Public Dataset by Olist, Kaggle. The derived CSV result tables follow the dataset's [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/) license.
"""

report_path = results_dir / "results_questions_3_4_5.md"
report_path.write_text(report, encoding="utf-8")
print("Saved result files:")
for filename in (*output_names.values(), report_path.name):
    print(f"resultados/{filename}")

Saved result files:
resultados/question_3_states.csv
resultados/question_4_categories.csv
resultados/question_5_revenue.csv
resultados/results_questions_3_4_5.md
